In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
#  🦴 Bone Fracture 10-Class Classification – Lightweight (< 1 min)
#  MobileNetV2 · img 128 · 3 epochs
# ============================================================
import os, pathlib, time, json as _json, warnings, numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime
warnings.filterwarnings('ignore')

# ── 경로 설정 ────────────────────────────────────────────────
INPUT_PATH  = "/content/drive/MyDrive/2026_lecture/Medical_AI/Medical_Imagining/Bone Break Classification"
OUTPUT_BASE = "/content/drive/MyDrive/2026_lecture/Medical_AI/1week/output"
RUN_STAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = os.path.join(OUTPUT_BASE, f"run_{RUN_STAMP}")
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"✅ Output: {OUTPUT_PATH}")

# ── 하이퍼파라미터 (경량) ────────────────────────────────────
IMG_SIZE, BATCH_SIZE, NUM_EPOCHS, LR, SEED = 128, 64, 3, 1e-3, 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"⚡ Device: {DEVICE}")

IMG_EXTS = {'.png','.jpg','.jpeg','.bmp','.tiff','.tif','.webp'}

# ── 커스텀 ImageFolder (모든 이미지 확장자) ──────────────────
class MultiExtImageFolder(ImageFolder):
    def is_valid_file(self, path):
        return pathlib.Path(path).suffix.lower() in IMG_EXTS

# ── 폴더 구조 분석 ──────────────────────────────────────────
class_names = sorted([d for d in os.listdir(INPUT_PATH) if os.path.isdir(os.path.join(INPUT_PATH, d))])
print(f"📂 Classes({len(class_names)}): {class_names}")
class_counts = {}
for cls in class_names:
    count = len([f for f in pathlib.Path(os.path.join(INPUT_PATH, cls)).rglob("*") if f.suffix.lower() in IMG_EXTS])
    class_counts[cls] = count
    print(f"  [{cls:30s}] {count} imgs")
total_images = sum(class_counts.values())
print(f"  Total: {total_images}")

# ── 클래스 분포 차트 ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12,5))
bars = ax.bar(range(len(class_names)), [class_counts[c] for c in class_names], color=plt.cm.Set3.colors[:len(class_names)])
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel("Count"); ax.set_title("📊 Class Distribution", fontweight='bold')
for b,c in zip(bars, class_names): ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, str(class_counts[c]), ha='center', fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH,'01_class_distribution.png'),dpi=100); plt.show()

# ── transforms ───────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ── 데이터 분할 (80/10/10) ───────────────────────────────────
full_ds = MultiExtImageFolder(root=INPUT_PATH)
n = len(full_ds)
indices = list(range(n)); np.random.shuffle(indices)
n_train = int(n*0.8); n_val = int(n*0.1)
train_idx, val_idx, test_idx = indices[:n_train], indices[n_train:n_train+n_val], indices[n_train+n_val:]

train_ds = Subset(MultiExtImageFolder(root=INPUT_PATH, transform=train_tf), train_idx)
val_ds   = Subset(MultiExtImageFolder(root=INPUT_PATH, transform=eval_tf), val_idx)
test_ds  = Subset(MultiExtImageFolder(root=INPUT_PATH, transform=eval_tf), test_idx)
print(f"\n📦 Train:{len(train_ds)} | Val:{len(val_ds)} | Test:{len(test_ds)}")

# WeightedRandomSampler
labels_train = [full_ds.targets[i] for i in train_idx]
cls_count = np.array([labels_train.count(i) for i in range(len(class_names))])
w = 1.0 / np.maximum(cls_count, 1)
sample_w = np.array([w[l] for l in labels_train])
sampler = WeightedRandomSampler(torch.from_numpy(sample_w).float(), len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── 모델: MobileNetV2 (경량) ─────────────────────────────────
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, len(class_names))
model = model.to(DEVICE)
print(f"🧠 Model: MobileNetV2 | Params: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ── 학습 ─────────────────────────────────────────────────────
history = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]}
best_val_acc = 0.0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS+1):
    model.train(); t_loss, t_ok, t_n = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        t_loss += loss.item()*imgs.size(0); t_ok += (out.argmax(1)==labels).sum().item(); t_n += imgs.size(0)

    model.eval(); v_loss, v_ok, v_n = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs); loss = criterion(out, labels)
            v_loss += loss.item()*imgs.size(0); v_ok += (out.argmax(1)==labels).sum().item(); v_n += imgs.size(0)
    scheduler.step()

    tl, ta = t_loss/t_n, t_ok/t_n*100; vl, va = v_loss/v_n, v_ok/v_n*100
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl); history['val_acc'].append(va)
    flag = ""
    if va > best_val_acc:
        best_val_acc = va; torch.save(model.state_dict(), os.path.join(OUTPUT_PATH,'best_model.pth')); flag = " ⭐"
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss:{tl:.4f} Acc:{ta:.1f}% | Val Loss:{vl:.4f} Acc:{va:.1f}%{flag}")

elapsed = time.time()-t0
print(f"\n✅ Training done in {elapsed:.1f}s | Best Val Acc: {best_val_acc:.1f}%")

# ── 학습 곡선 ────────────────────────────────────────────────
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
ep = range(1, len(history['train_loss'])+1)
ax1.plot(ep, history['train_loss'], 'b-o', markersize=4, label='Train')
ax1.plot(ep, history['val_loss'], 'r-o', markersize=4, label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xlabel('Epoch'); ax1.grid(alpha=0.3)
ax2.plot(ep, history['train_acc'], 'b-o', markersize=4, label='Train')
ax2.plot(ep, history['val_acc'], 'r-o', markersize=4, label='Val')
ax2.set_title('Accuracy (%)'); ax2.legend(); ax2.set_xlabel('Epoch'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH,'02_training_curve.png'),dpi=100); plt.show()

# ── 테스트 평가 ──────────────────────────────────────────────
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH,'best_model.pth')))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.numpy())
all_preds, all_labels = np.array(all_preds), np.array(all_labels)
test_acc = (all_preds==all_labels).mean()*100
print(f"\n📊 Test Accuracy: {test_acc:.2f}%")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

# ── Confusion Matrix ─────────────────────────────────────────
short = [c.replace(" fracture","").replace(" Fracture","") for c in class_names]
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=short, yticklabels=short, ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Confusion Matrix (Test Acc: {test_acc:.2f}%)', fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH,'03_confusion_matrix.png'),dpi=100); plt.show()

# ── Per-Class Accuracy ───────────────────────────────────────
per_acc = cm.diagonal()/cm.sum(axis=1)*100
fig, ax = plt.subplots(figsize=(12,5))
colors = ['#4CAF50' if a>=80 else '#FF9800' if a>=60 else '#F44336' for a in per_acc]
bars = ax.bar(range(len(class_names)), per_acc, color=colors)
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Accuracy (%)'); ax.set_title('Per-Class Test Accuracy', fontweight='bold')
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5); ax.set_ylim(0,115)
for b,a in zip(bars, per_acc): ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{a:.1f}%', ha='center', fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH,'05_per_class_accuracy.png'),dpi=100); plt.show()

# ── 요약 저장 ────────────────────────────────────────────────
summary = {
    "run_timestamp": RUN_STAMP, "model": "MobileNetV2",
    "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS, "lr": LR,
    "training_time_sec": round(elapsed,1),
    "best_val_acc": round(best_val_acc,2), "test_acc": round(test_acc,2),
    "num_classes": len(class_names), "class_names": class_names,
    "dataset_total": total_images,
    "train_samples": len(train_ds), "val_samples": len(val_ds), "test_samples": len(test_ds),
    "per_class_acc": {c: round(float(a),2) for c,a in zip(class_names, per_acc)},
}
with open(os.path.join(OUTPUT_PATH,'06_run_summary.json'),'w') as f:
    _json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"🎉 완료! Output: {OUTPUT_PATH}")
print(f"   01_class_distribution.png")
print(f"   02_training_curve.png")
print(f"   03_confusion_matrix.png")
print(f"   05_per_class_accuracy.png")
print(f"   06_run_summary.json")
print(f"   best_model.pth")
print(f"{'='*50}")
print(f"✅ Best Val Acc: {best_val_acc:.1f}%  |  Test Acc: {test_acc:.1f}%")


✅ Output: /content/drive/MyDrive/2026_lecture/Medical_AI/1week/output/run_20260414_045350
⚡ Device: cuda
📂 Classes(10): ['Avulsion fracture', 'Comminuted fracture', 'Fracture Dislocation', 'Greenstick fracture', 'Hairline Fracture', 'Impacted fracture', 'Longitudinal fracture', 'Oblique fracture', 'Pathological fracture', 'Spiral Fracture']
  [Avulsion fracture             ] 123 imgs
  [Comminuted fracture           ] 148 imgs
  [Fracture Dislocation          ] 156 imgs
  [Greenstick fracture           ] 122 imgs
  [Hairline Fracture             ] 111 imgs
  [Impacted fracture             ] 84 imgs
  [Longitudinal fracture         ] 80 imgs
  [Oblique fracture              ] 85 imgs
  [Pathological fracture         ] 134 imgs
  [Spiral Fracture               ] 86 imgs
  Total: 1129

📦 Train:903 | Val:112 | Test:114
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 93.0MB/s]


🧠 Model: MobileNetV2 | Params: 2,236,682
Epoch 1/3 | Train Loss:2.0904 Acc:27.7% | Val Loss:2.2062 Acc:17.9% ⭐
Epoch 2/3 | Train Loss:1.6366 Acc:46.1% | Val Loss:1.9378 Acc:33.0% ⭐
Epoch 3/3 | Train Loss:1.3385 Acc:56.4% | Val Loss:1.9707 Acc:36.6% ⭐

✅ Training done in 140.1s | Best Val Acc: 36.6%

📊 Test Accuracy: 35.96%
                       precision    recall  f1-score   support

    Avulsion fracture      0.417     0.455     0.435        11
  Comminuted fracture      0.400     0.267     0.320        15
 Fracture Dislocation      0.400     0.118     0.182        17
  Greenstick fracture      0.368     0.700     0.483        10
    Hairline Fracture      0.214     0.273     0.240        11
    Impacted fracture      0.500     0.500     0.500         8
Longitudinal fracture      0.222     0.167     0.190        12
     Oblique fracture      0.333     0.273     0.300        11
Pathological fracture      0.400     0.667     0.500        12
      Spiral Fracture      0.375     0.429  